In [1]:
from pathlib import Path
import sys
import math
import csv
from collections import Counter
import copy

DATA_DIR = Path("data")
print("Python 版本：", sys.version.split()[0])
print("当前目录：", Path.cwd())
print("数据目录：", DATA_DIR.resolve())
for name in ("train.txt", "dev.csv", "test.csv", "challenge.csv"):
    print(name, "已找到" if (DATA_DIR / name).is_file() else "未找到，请检查解压位置或工作目录")

Python 版本： 3.10.21
当前目录： C:\Users\13590
数据目录： C:\Users\13590\data
train.txt 已找到
dev.csv 已找到
test.csv 已找到
challenge.csv 已找到


In [7]:
class NGramLM:
    def __init__(self, n:int):
        """
        n=1 Unigram；n=2 Bigram；n=3 Trigram
        约定：
        1.训练句子为词语列表，不含任何特殊标记；训练时自动补句首<s>、句末</s>
        2.Unigram不补句首<s>；Bigram补1个<s>；Trigram补2个<s>
        3.预测词表V：普通词 + </s> + <unk>；不含<s>；<s>仅做上下文
        """
        self.n = n
        self.ngram_counts = Counter()          # key:tuple(history + word)
        self.history_counts = Counter()         # key:tuple history；该历史后面预测词总次数
        self.normal_vocab = set()               # 训练普通词语，不含特殊标记
        self.vocab = set()                      # 预测词表
        self.total_unigram = 0                  # unigram总词次（含</s>，不含<s>）

    def add_sentence(self, word_list:list):
        """输入普通词语列表，添加一句训练，自动加边界标记"""
        for w in word_list:
            self.normal_vocab.add(w)
        tokens = copy.deepcopy(word_list)
        tokens.append("</s>")
        if self.n >=2:
            tokens = ["<s>"]*(self.n-1) + tokens
        if self.n == 1:
            for w in tokens:
                self.ngram_counts[(w,)] +=1
                self.total_unigram +=1
        else:
            for i in range(len(tokens)- self.n +1):
                ngram = tuple(tokens[i:i+self.n])
                history = ngram[:-1]
                target_word = ngram[-1]
                self.ngram_counts[ngram] +=1
                self.history_counts[history] +=1

    def build_vocab(self):
        """训练完成后调用，构建预测词表V"""
        self.vocab = set(self.normal_vocab)
        self.vocab.add("</s>")
        self.vocab.add("<unk>")

    def map_oov(self, word):
        """OOV映射，<s>保留原样，未知词映射<unk>"""
        if word == "<s>":
            return "<s>"
        if word in self.normal_vocab:
            return word
        return "<unk>"

    def get_cond_prob(self, history_tuple:tuple, target_word, k_smooth=0.0):
        """加K平滑；k=0为MLE未平滑"""
        V_size = len(self.vocab)
        h = history_tuple
        w = target_word
        c_hw = self.ngram_counts.get(tuple(list(h)+[w]), 0)
        c_h = self.history_counts.get(h, 0)

        if self.n == 1:
            c_hw = self.ngram_counts.get((w,), 0)
            c_h = self.total_unigram

        if k_smooth <= 1e-12:
            if c_h == 0:
                return 0.0, "no_history"
            prob = c_hw / c_h
            return prob, "ok"
        else:
            numerator = c_hw + k_smooth
            denominator = c_h + k_smooth * V_size
            prob = numerator / denominator
            return prob, "ok"

    def score_sentence_log(self, word_list:list, k_smooth=0.0):
        """输入普通词语列表；返回 logP(S) 和每一步明细，包含预测</s>"""
        mapped = [self.map_oov(w) for w in word_list]
        mapped.append("</s>")
        if self.n >=2:
            mapped = ["<s>"]*(self.n-1) + mapped
        steps = []
        total_log = 0.0
        for i in range(self.n-1, len(mapped)):
            history = tuple(mapped[i-(self.n-1):i])
            target = mapped[i]
            p, status = self.get_cond_prob(history, target, k_smooth=k_smooth)
            if abs(p) < 1e-15:
                logp = -math.inf
            else:
                logp = math.log(p)
            steps.append({"history":history, "target":target, "prob":p, "log_prob":logp, "status":status})
            if math.isinf(logp):
                total_log = -math.inf
            else:
                if not math.isinf(total_log):
                    total_log += logp
        return total_log, steps

# 插值平滑类（任务一子任务3使用）
class InterpolateLM:
    def __init__(self, lm_uni:NGramLM, lm_bi:NGramLM, lm_tri:NGramLM, lam1, lam2, lam3):
        self.lm_uni = lm_uni
        self.lm_bi = lm_bi
        self.lm_tri = lm_tri
        self.lam1 = lam1
        self.lam2 = lam2
        self.lam3 = lam3
        self.vocab = lm_uni.vocab

    def score_sentence_log(self, word_list:list, k_uni=0.1, k_bi=0.1, k_tri=0.1):
        mapped = [self.lm_uni.map_oov(w) for w in word_list]
        mapped.append("</s>")
        mapped_tri = ["<s>"]*2 + mapped
        steps = []
        total_log =0.0
        for i in range(2, len(mapped_tri)):
            target = mapped_tri[i]
            h_tri = tuple(mapped_tri[i-2:i])
            p_tri,_ = self.lm_tri.get_cond_prob(h_tri, target, k_smooth=k_tri)
            h_bi = (mapped_tri[i-1], )
            p_bi,_ = self.lm_bi.get_cond_prob(h_bi, target, k_smooth=k_bi)
            h_uni = tuple()
            p_uni,_ = self.lm_uni.get_cond_prob(h_uni, target, k_smooth=k_uni)
            p_mix = self.lam1 * p_uni + self.lam2 * p_bi + self.lam3 * p_tri
            logp = math.log(p_mix)
            steps.append({"history_tri":h_tri, "target":target, "p_uni":p_uni, "p_bi":p_bi, "p_tri":p_tri, "p_mix":p_mix, "log_prob":logp})
            if math.isinf(total_log):
                continue
            total_log += logp
        return total_log, steps
        

In [8]:
toy_text = """
学校 今天 发布 通知
学校 明天 发布 通知
学生 今天 提交 报告
学生 明天 提交 报告
教师 今天 发布 作业
教师 明天 提交 作业
"""

# 解析文本成句子列表
toy_sentences = []
for line in toy_text.strip().splitlines():
    words = line.strip().split()
    toy_sentences.append(words)

print("===按句组织的词语列表===")
for s in toy_sentences:
    print(s)

# 训练3个阶数模型
lm_uni_toy = NGramLM(n=1)
lm_bi_toy = NGramLM(n=2)
lm_tri_toy = NGramLM(n=3)
for sent in toy_sentences:
    lm_uni_toy.add_sentence(sent)
    lm_bi_toy.add_sentence(sent)
    lm_tri_toy.add_sentence(sent)

lm_uni_toy.build_vocab()
lm_bi_toy.build_vocab()
lm_tri_toy.build_vocab()

print("\n普通词表大小：", len(lm_uni_toy.normal_vocab))
print("预测词表大小：", len(lm_uni_toy.vocab))

# 查询指定计数
print("\n【查询1】单个词 \"发布\" 计数：", lm_uni_toy.ngram_counts.get(("发布",), 0))
print("【查询2】Bigram \"发布 通知\" 计数：", lm_bi_toy.ngram_counts.get(("发布","通知"),0))
print("【查询3】Trigram \"今天 发布 通知\" 计数：", lm_tri_toy.ngram_counts.get(("今天","发布","通知"),0))

# 你可以在这里修改句子观察计数变化，实验报告记录变化
# toy_sentences_mod = toy_sentences + [["教师","今天","提交","通知"]]

===按句组织的词语列表===
['学校', '今天', '发布', '通知']
['学校', '明天', '发布', '通知']
['学生', '今天', '提交', '报告']
['学生', '明天', '提交', '报告']
['教师', '今天', '发布', '作业']
['教师', '明天', '提交', '作业']

普通词表大小： 10
预测词表大小： 12

【查询1】单个词 "发布" 计数： 3
【查询2】Bigram "发布 通知" 计数： 2
【查询3】Trigram "今天 发布 通知" 计数： 1


In [9]:
# 三个条件概率查询（未平滑 k=0）
p1, s1 = lm_bi_toy.get_cond_prob(("今天",), "发布", k_smooth=0)
print("今天 → 发布 | 概率=", p1, "状态=", s1)

p2, s2 = lm_bi_toy.get_cond_prob(("发布",), "通知", k_smooth=0)
print("发布 → 通知 | 概率=", p2, "状态=", s2)

p3, s3 = lm_tri_toy.get_cond_prob(("学校","今天"), "发布", k_smooth=0)
print("学校 今天 → 发布 | 概率=", p3, "状态=", s3)

# 两个测试句子
sent_a = ["学校", "今天", "发布", "通知"]
sent_b = ["学校", "今天", "通知", "发布"]

print("\n========句子A：学校 今天 发布 通知 逐项Bigram明细========")
log_a_bi, step_a_bi = lm_bi_toy.score_sentence_log(sent_a, k_smooth=0)
print(f"Bigram总对数得分 = {log_a_bi}")
for step in step_a_bi:
    print(step)

print("\n========句子B：学校 今天 通知 发布========")
log_b_bi, _ = lm_bi_toy.score_sentence_log(sent_b, k_smooth=0)
print(f"Bigram总对数得分 = {log_b_bi}")

log_a_uni,_ = lm_uni_toy.score_sentence_log(sent_a,0)
log_b_uni,_ = lm_uni_toy.score_sentence_log(sent_b,0)
log_a_tri,_ = lm_tri_toy.score_sentence_log(sent_a,0)
log_b_tri,_ = lm_tri_toy.score_sentence_log(sent_b,0)

print(f"\nUnigram: A={log_a_uni}  B={log_b_uni}")
print(f"Trigram: A={log_a_tri}  B={log_b_tri}")

今天 → 发布 | 概率= 0.6666666666666666 状态= ok
发布 → 通知 | 概率= 0.6666666666666666 状态= ok
学校 今天 → 发布 | 概率= 1.0 状态= ok

========句子A：学校 今天 发布 通知 逐项Bigram明细========
Bigram总对数得分 = -2.602689685444384
{'history': ('<s>',), 'target': '学校', 'prob': 0.3333333333333333, 'log_prob': -1.0986122886681098, 'status': 'ok'}
{'history': ('学校',), 'target': '今天', 'prob': 0.5, 'log_prob': -0.6931471805599453, 'status': 'ok'}
{'history': ('今天',), 'target': '发布', 'prob': 0.6666666666666666, 'log_prob': -0.40546510810816444, 'status': 'ok'}
{'history': ('发布',), 'target': '通知', 'prob': 0.6666666666666666, 'log_prob': -0.40546510810816444, 'status': 'ok'}
{'history': ('通知',), 'target': '</s>', 'prob': 1.0, 'log_prob': 0.0, 'status': 'ok'}

========句子B：学校 今天 通知 发布========
Bigram总对数得分 = -inf

Unigram: A=-11.63070850062661  B=-11.63070850062661
Trigram: A=-2.4849066497880004  B=-inf


In [10]:
# 测试观察零概率的两句
test_s1 = ["学校","今天","提交","报告"]
test_s2 = ["学校","今天","通知","报告"]

print("====不同k值平滑，句子1 log得分====")
for k in [0, 0.1, 0.5, 1.0]:
    logp, _ = lm_bi_toy.score_sentence_log(test_s1, k_smooth=k)
    print(f"k={k}  logP = {logp:.4f}")

# 构建插值模型 λ1=0.2 λ2=0.3 λ3=0.5 和为1
interp_lm = InterpolateLM(lm_uni_toy, lm_bi_toy, lm_tri_toy, lam1=0.2, lam2=0.3, lam3=0.5)
log_interp,_ = interp_lm.score_sentence_log(test_s1, k_uni=0.1, k_bi=0.1, k_tri=0.1)
print(f"\n插值模型句子1 logP = {log_interp:.4f}")

# 归一化检查1：已见历史 ("今天",) k=1
history_seen = ("今天",)
sum_p = 0.0
for w in lm_bi_toy.vocab:
    prob,_ = lm_bi_toy.get_cond_prob(history_seen, w, k_smooth=1.0)
    sum_p += prob
print(f"\n已见历史('今天',) k=1，全部词概率和 = {sum_p}")

# 归一化检查2：未见历史
history_unseen = ("<unk>", )
sum_p2 =0.0
for w in lm_bi_toy.vocab:
    prob,_ = lm_bi_toy.get_cond_prob(history_unseen, w, k_smooth=1.0)
    sum_p2 += prob
print(f"未见历史('<unk>',) k=1，全部词概率和 = {sum_p2}")

====不同k值平滑，句子1 log得分====
k=0  logP = -3.2958
k=0.1  logP = -4.7541
k=0.5  logP = -7.4784
k=1.0  logP = -8.9025

插值模型句子1 logP = -6.1867

已见历史('今天',) k=1，全部词概率和 = 1.0
未见历史('<unk>',) k=1，全部词概率和 = 1.0


In [11]:
eval_sentences = [
    ["学校","明天","提交","报告"],
    ["学生","今天","发布","作业"],
    ["教师","今天","提交","报告"],
    ["学生","明天","发布","通知"]
]

model_cfg = [
    ("Unigram 加一", lm_uni_toy, 1.0),
    ("Bigram 加一", lm_bi_toy, 1.0),
    ("Trigram 加一", lm_tri_toy, 1.0),
    ("Bigram k=0.1", lm_bi_toy, 0.1),
    ("Bigram k=0.5", lm_bi_toy, 0.5),
]

print(f"{'模型配置':14s} | 句子1 | 句子2 | 句子3 | 句子4")
for name, lm, k in model_cfg:
    score_list = []
    for s in eval_sentences:
        lp,_ = lm.score_sentence_log(s, k_smooth=k)
        score_list.append(f"{lp:.3f}")
    print(f"{name:14s} | {' | '.join(score_list)}")

模型配置           | 句子1 | 句子2 | 句子3 | 句子4
Unigram 加一     | -11.773 | -11.773 | -11.773 | -11.773
Bigram 加一      | -8.497 | -8.902 | -8.902 | -8.902
Trigram 加一     | -9.789 | -10.120 | -9.715 | -9.715
Bigram k=0.1   | -4.107 | -4.754 | -4.754 | -4.754
Bigram k=0.5   | -6.968 | -7.478 | -7.478 | -7.478


In [12]:
demo_raw = ["工作人员","正在","____","最新","通知"]
candidates = ["提交","发布","召开","参加"]

def fill_blank(sent_raw, blank_word):
    return [blank_word if x=="____" else x for x in sent_raw]

print("===缺词替换演示===")
for c in candidates:
    filled = fill_blank(demo_raw, c)
    print(f"填入 {c} → {filled}")

print("\n原理说明：")
print("Bigram：随候选变化两项 P(c|工作人员,正在)、P(最新|c)；后面“最新→通知”同题四个候选不变。")
print("Trigram：额外利用 P(通知 | c, 最新)")

===缺词替换演示===
填入 提交 → ['工作人员', '正在', '提交', '最新', '通知']
填入 发布 → ['工作人员', '正在', '发布', '最新', '通知']
填入 召开 → ['工作人员', '正在', '召开', '最新', '通知']
填入 参加 → ['工作人员', '正在', '参加', '最新', '通知']

原理说明：
Bigram：随候选变化两项 P(c|工作人员,正在)、P(最新|c)；后面“最新→通知”同题四个候选不变。
Trigram：额外利用 P(通知 | c, 最新)


In [13]:
def load_train_txt(path):
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            words = line.split()
            sentences.append(words)
    return sentences

def load_csv_question(path):
    qs = []
    with open(path, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            qs.append(row)
    return qs

train_sents = load_train_txt(DATA_DIR / "train.txt")
dev_questions = load_csv_question(DATA_DIR / "dev.csv")
test_questions = load_csv_question(DATA_DIR / "test.csv")
challenge_questions = load_csv_question(DATA_DIR / "challenge.csv")

# train.txt统计校验
word_total = sum(len(s) for s in train_sents)
normal_vocab = set()
for s in train_sents:
    for w in s:
        normal_vocab.add(w)

print("==== train.txt校验 ====")
print(f"训练句子数：{len(train_sents)}")
print(f"普通词总词次：{word_total}")
print(f"普通词种数：{len(normal_vocab)}")
print(f"dev题：{len(dev_questions)}  test题：{len(test_questions)}  challenge题：{len(challenge_questions)}")
print("文档预期：词次358，词种141；核对输出是否一致")

==== train.txt校验 ====
训练句子数：72
普通词总词次：358
普通词种数：141
dev题：12  test题：12  challenge题：6
文档预期：词次358，词种141；核对输出是否一致


In [14]:
lm_uni_proj = NGramLM(n=1)
lm_bi_proj = NGramLM(n=2)
lm_tri_proj = NGramLM(n=3)

for sent in train_sents:
    lm_uni_proj.add_sentence(sent)
    lm_bi_proj.add_sentence(sent)
    lm_tri_proj.add_sentence(sent)

lm_uni_proj.build_vocab()
lm_bi_proj.build_vocab()
lm_tri_proj.build_vocab()

print(f"项目预测词表大小 uni:{len(lm_uni_proj.vocab)}")
print("文档预期预测词表大小=143，请核对")

项目预测词表大小 uni:143
文档预期预测词表大小=143，请核对


In [15]:
TOL = 1e-10  # 并列判断容差

def solve_one_question(q, lm:NGramLM, k_smooth):
    sent_raw = q["sentence"].split()
    opts = [q["option_A"], q["option_B"], q["option_C"], q["option_D"]]
    opt_labels = ["A","B","C","D"]
    score_list = []
    for lab, w in zip(opt_labels, opts):
        filled_sent = [w if x=="____" else x for x in sent_raw]
        log_score, _ = lm.score_sentence_log(filled_sent, k_smooth=k_smooth)
        score_list.append((lab, w, log_score))

    score_sorted = sorted(score_list, key=lambda x:x[2], reverse=True)
    max_sc = score_sorted[0][2]
    tie_items = [item for item in score_list if abs(item[2] - max_sc) <= TOL]
    is_tie = len(tie_items) > 1
    tie_labels = sorted([x[0] for x in tie_items])
    pred_label = tie_labels[0]

    return {
        "id": q["id"],
        "predicted_option": pred_label,
        "predicted_word": [x[1] for x in score_list if x[0]==pred_label][0],
        "score_A": score_list[0][2],
        "score_B": score_list[1][2],
        "score_C": score_list[2][2],
        "score_D": score_list[3][2],
        "ranking": ",".join([f"{lab}:{sc:.3f}" for lab,wd,sc in score_sorted]),
        "is_tie": is_tie,
        "tied_options": tie_labels
    }

# 简单测试dev第一题
res_test = solve_one_question(dev_questions[0], lm_bi_proj, k_smooth=1.0)
print("单题测试输出：")
print(res_test)

单题测试输出：
{'id': 'D01', 'predicted_option': 'A', 'predicted_word': '发布', 'score_A': -25.33199812485973, 'score_B': -26.025145305419674, 'score_C': -26.025145305419674, 'score_D': -26.718292485979617, 'ranking': 'A:-25.332,B:-26.025,C:-26.025,D:-26.718', 'is_tie': False, 'tied_options': ['A']}


In [16]:
def eval_dataset(question_list, lm, k):
    correct = 0
    tie_count = 0
    total = len(question_list)
    for q in question_list:
        out = solve_one_question(q, lm, k_smooth=k)
        ref_ans = q["answer"]
        if out["is_tie"]:
            tie_count += 1
        if out["predicted_option"] == ref_ans:
            correct += 1
    acc = correct / total
    return {"correct":correct, "total":total, "acc":acc, "tie":tie_count}

config_list = [
    ("Unigram add‑1", lm_uni_proj, 1.0),
    ("Bigram add‑1", lm_bi_proj, 1.0),
    ("Bigram k=0.1", lm_bi_proj, 0.1),
    ("Bigram k=0.5", lm_bi_proj, 0.5),
]

print(f"{'配置':16s} | 答对数 | 准确率 | 并列题数")
for cfg_name, lm, k in config_list:
    ret = eval_dataset(dev_questions, lm, k)
    print(f"{cfg_name:16s} | {ret['correct']:2d}/{ret['total']} | {ret['acc']:.3f} | {ret['tie']:2d}")


配置               | 答对数 | 准确率 | 并列题数
Unigram add‑1    |  3/12 | 0.250 | 12
Bigram add‑1     | 12/12 | 1.000 |  1
Bigram k=0.1     | 12/12 | 1.000 |  1
Bigram k=0.5     | 12/12 | 1.000 |  1
